## Adaptive RAG

### 2.1 Setup (LLM, embeddings, vectorstores)

In [13]:
# %pip install -r r.txt

In [14]:
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import time
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
INDEX_NAME = "coffee-hybrid"
index_name_for_dense = INDEX_NAME

existing = [d["name"] for d in pc.list_indexes()]
if INDEX_NAME in existing:
    try:
        pc.delete_index(name=INDEX_NAME)
        while INDEX_NAME in [d["name"] for d in pc.list_indexes()]:
            time.sleep(1)
        print("✅ Deleted.")
    except Exception as e:
        print("Warning: failed to delete existing index:", e)

pc.create_index(
    name=INDEX_NAME,
    dimension=384,
    metric='cosine',
    spec=ServerlessSpec(cloud='aws', region="us-east-1")
)

# Wait until index is ready for writes
print("⏳ Waiting for index to be ready...")
while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)
print("✅ Index ready.")

✅ Deleted.
⏳ Waiting for index to be ready...
✅ Index ready.


In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
)

In [16]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document

vs_docs = PineconeVectorStore(index_name=index_name_for_dense, embedding=embeddings, text_key="text")
USER_ID = "user_001"
MEM_NS  = f"mem_{USER_ID}"
vs_mem  = PineconeVectorStore(index_name=index_name_for_dense, embedding=embeddings, text_key="text", namespace=MEM_NS)

# Helper to store long-term memory
def add_memory(text: str, **meta):
    meta.setdefault("kind", "memory"); meta.setdefault("ts", int(datetime.utcnow().timestamp()))
    vs_mem.add_documents([Document(page_content=text, metadata=meta)])


In [17]:
# STEP 2.0 — Document Ingestion (Load, Clean, Store)

from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import re

# Function to remove boilerplate disclaimer from document content
def remove_disclaimer(text):
    """Remove common disclaimer text that appears on all pages."""
    disclaimer_pattern = r"Disclaimer:[\s\S]*?before trying new herbs or routines\.\s*"
    text = re.sub(disclaimer_pattern, "", text, flags=re.IGNORECASE)
    return text.strip()

# Load HTML documents from coffee_pages directory
html_dir = Path("coffee_pages")
html_docs = []

for fp in html_dir.glob("*.html"):
    try:
        loaded = UnstructuredHTMLLoader(str(fp)).load()
        for d in loaded:
            meta = dict(d.metadata or {})
            meta.pop("text", None)
            meta["source"] = str(fp)
            
            # Remove disclaimer from content
            content = remove_disclaimer(d.page_content)
            
            if content:  # Only keep if there's meaningful content left
                html_docs.append(Document(page_content=content, metadata=meta))
    except Exception as e:
        print(f"[WARN] Skipping {fp.name}: {e}")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
chunks = splitter.split_documents(html_docs)

# Store in Pinecone using PineconeVectorStore
vs_docs.add_documents(chunks)

print(f"✅ Stored {len(chunks)} chunks in '{INDEX_NAME}' (disclaimer removed)")

✅ Stored 53 chunks in 'coffee-hybrid' (disclaimer removed)


### 2.2 Build retrievers (dense, hybrid, memory) + multi-query & MMR

In [18]:
# STEP 2 — Retrievers (dense / hybrid / memory) + MQ + MMR

# Dense (MMR for diversity)
ret_dense_mmr = vs_docs.as_retriever(search_type="mmr", search_kwargs={"k": 20, "fetch_k": 60, "lambda_mult": 0.5})

# Hybrid (dense + sparse) via LangChain community retriever
from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever
pc_index = pc.Index(INDEX_NAME)
bm25 = BM25Encoder().default()  # (fit on corpus if you still have it: bm25.fit([c.page_content for c in chunks]))
ret_hybrid = PineconeHybridSearchRetriever(
    embeddings=embeddings, 
    sparse_encoder=bm25, 
    index=pc_index, 
    top_k=20, 
    alpha=0.5,
    text_key="text"  # match the text_key used when storing documents
)

# Memory retriever (dense)
ret_mem = vs_mem.as_retriever(search_kwargs={"k": 6, "namespace": MEM_NS})

# Multi-query wrapper (over the docs retrievers)
from langchain.retrievers.multi_query import MultiQueryRetriever
ret_dense_mq  = MultiQueryRetriever.from_llm(retriever=ret_dense_mmr, llm=llm, include_original=True)
ret_hybrid_mq = MultiQueryRetriever.from_llm(retriever=ret_hybrid,   llm=llm, include_original=True)

### 2.3 Router (prompt → JSON) decides strategy; fuse & dedup

In [19]:
# STEP 3 — Adaptive router + fusion + dedup compression

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.document_compressors import DocumentCompressorPipeline

router_parser = JsonOutputParser()
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a router. Choose the best retrieval strategy based on the QUESTION.\n"
     "Return ONLY JSON with keys:\n"
     "  strategy: one of ['dense','hybrid','dense_mq','hybrid_mq','memory_only','dense_plus_memory','hybrid_plus_memory']\n"
     "  k: integer between 8 and 30\n"
     "  alpha: number 0..1 for hybrid blending (ignored if not hybrid)\n"
     "  notes: short string\n"
     "{format}"),
    ("human", "{question}")
])
router_chain = router_prompt | llm | router_parser

# Build a retriever based on router decision (LangChain tools only)
def make_retriever(decision: dict):
    strat = decision.get("strategy", "hybrid_mq")
    k     = int(decision.get("k", 20))
    alpha_val = decision.get("alpha")
    alpha = float(alpha_val) if alpha_val is not None else 0.5

    # tune top_k / alpha where applicable
    if strat in {"hybrid", "hybrid_mq"}:
        ret_hybrid.top_k = k
        ret_hybrid.alpha = alpha
        base = ret_hybrid_mq if strat == "hybrid_mq" else ret_hybrid
    elif strat in {"dense","dense_mq"}:
        # ret_dense_mmr already has MMR; MQ adds paraphrases
        base = ret_dense_mq if strat == "dense_mq" else ret_dense_mmr
    elif strat == "memory_only":
        base = ret_mem
    elif strat == "dense_plus_memory":
        base = EnsembleRetriever(retrievers=[ret_dense_mq, ret_mem], weights=[0.75, 0.25])
    elif strat == "hybrid_plus_memory":
        # Use hybrid multi-query + memory
        base = EnsembleRetriever(retrievers=[ret_hybrid_mq, ret_mem], weights=[0.75, 0.25])
    else:
        base = ret_hybrid_mq

    # de-dup near-identical chunks
    dedup = EmbeddingsRedundantFilter(embeddings=embeddings, similarity_threshold=0.92)
    compressor = DocumentCompressorPipeline(transformers=[dedup])

    return ContextualCompressionRetriever(base_retriever=base, base_compressor=compressor)

### 2.4 Answer prompt (placeholders) + small helpers

In [20]:
# STEP 4 — Answer prompt with placeholders

from langchain_core.prompts import MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import HumanMessage, AIMessage

def format_block(docs):
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source") or d.metadata.get("file_name") or d.metadata.get("kind") or f"S{i}"
        snip = (d.page_content or "").strip().replace("\n", " ")
        lines.append(f"[S{i}] {src}\n{snip}")
    return "\n\n".join(lines)

def format_memory(docs):
    """Format memory docs as [M#] snippets."""
    lines = []
    for i, d in enumerate(docs, 1):
        snip = (d.page_content or "").strip().replace("\n", " ")
        lines.append(f"[M{i}] {snip}")
    return "\n".join(lines) if lines else "(no saved memories)"

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are Askly, a helpful assistant.\n"
     "Answer using CONTEXT (corpus documents) and USER MEMORY (personal preferences/facts saved by the user).\n"
     "If info is missing, say so. Cite corpus sources like [S1],[S2] and memory like [M1],[M2].\n"
     "When the user asks about their preferences, ALWAYS check USER MEMORY first.\n"
     "Be concise and actionable."
    ),
    MessagesPlaceholder("history"),
    ("system", "USER MEMORY:\n{memory}"),
    ("system", "CONTEXT:\n{context}"),
    ("human", "{question}")
])
answer_chain = answer_prompt | llm | StrOutputParser()

# simple token budget (approx)
def approx_tokens(s: str) -> int: return max(1, len(s or "") // 4)
def cap_docs(docs, max_tokens=1200):
    kept, total = [], 0
    for d in docs:
        n = approx_tokens(d.page_content)
        if total + n > max_tokens: break
        kept.append(d); total += n
    return kept

### 2.5 Adaptive Pipeline & Demo

An **adaptive retriever** is different from a fixed one:
- **Fixed retriever** — always uses the same strategy (e.g. always dense MMR, or always hybrid)
- **Adaptive retriever** — *routes* the query to different strategies based on its characteristics

The router LLM analyzes each question and decides:
- **"What is X?"** → `hybrid` (exact term + semantic search)
- **"What did you say about Y?"** → `dense_plus_memory` (history context needed)
- **"What do I prefer?"** → `memory_only` (pure fact recall)
- **"Compare X and Y?"** → `dense_mq` (needs multi-hop reasoning via rephrasing)

This is powerful because different questions need different retrieval strategies. For example, asking about a specific product ("ashwagandha") benefits from keyword precision (hybrid), but asking "compare" benefits from semantic paraphrasing (multi-query). And personal preferences live in memory, not the corpus.

---

In [21]:
# STEP 5 — Adaptive Pipeline + Scripted Demo

def adaptive_answer(question: str, history=None, verbose=True):
    """
    Adaptive RAG pipeline: route → retrieve → answer.
    
    Memory is ALWAYS fetched separately from ret_mem and passed
    into the dedicated {memory} placeholder in the prompt, so
    the LLM can distinguish personal facts from corpus docs.
    """
    history = history or []

    # 1) route — router decides strategy
    decision = router_chain.invoke({
        "question": question, 
        "format": router_parser.get_format_instructions()
    })

    if verbose:
        k = decision.get('k', '?')
        alpha = decision.get('alpha')
        alpha_str = f"{alpha:.1f}" if alpha is not None else "N/A"
        print(f"  🔀 ROUTER: strategy={decision.get('strategy', '?')}, k={k}, alpha={alpha_str}")
        print(f"     Notes: {decision.get('notes', '(no notes)')}")

    # 2) build retriever per decision
    retr = make_retriever(decision)

    # 3a) retrieve corpus docs via the adaptive retriever
    pool = retr.invoke(question)
    
    # 3b) ALWAYS fetch memory docs separately so they are never lost
    mem_docs = ret_mem.invoke(question)
    
    if verbose:
        print(f"  📊 RETRIEVAL: {len(pool)} corpus docs, {len(mem_docs)} memory docs")

    # 4) cap corpus docs to token budget
    final_corpus = cap_docs(pool, max_tokens=1200)
    
    if verbose:
        pre_cap  = sum(approx_tokens(d.page_content) for d in pool)
        post_cap = sum(approx_tokens(d.page_content) for d in final_corpus)
        pct = (100 * post_cap // pre_cap) if pre_cap > 0 else 0
        print(f"  📈 TOKEN BUDGET: {len(final_corpus)} corpus docs kept ({post_cap}/{pre_cap} tokens, {pct}%)")
        if mem_docs:
            for i, d in enumerate(mem_docs, 1):
                print(f"  🧠 MEMORY[{i}]: {d.page_content[:80]}")

    # 5) format context + memory separately, then answer
    ctx = format_block(final_corpus)
    mem = format_memory(mem_docs)
    
    answer = answer_chain.invoke({
        "history": history, 
        "context": ctx, 
        "memory": mem,
        "question": question
    })
    
    return decision, answer, final_corpus, mem_docs

# ── DEMO: 6-turn adaptive conversation ──────────────────────────────────

DEMO = [
    ("ask", "What is ashwagandha?"),
    ("ask", "What did you just tell me about it?"),
    ("mem", "I avoid caffeine because it makes me jittery"),
    ("ask", "Suggest a good morning drink for me"),
    ("ask", "How does mushroom coffee compare to what we discussed?"),
    ("ask", "What do I personally prefer?"),
]

history = []
print("\n🤖 ADAPTIVE RAG — Scripted Demo\n" + "=" * 70)

for kind, text in DEMO:
    
    # ── Memory save ──────────────────────────────────────────────────────
    if kind == "mem":
        print(f"\n📌 SAVING MEMORY: \"{text}\"")
        add_memory(text)
        continue
    
    # ── Question ─────────────────────────────────────────────────────────
    turn = len(history) // 2 + 1
    print(f"\n{'─' * 70}")
    print(f"  Turn {turn}  |  History: {len(history)} messages")
    print(f"  Q: {text}")
    print(f"{'─' * 70}")

    history.append(HumanMessage(content=text))

    # Adaptive answer with instrumentation
    decision, answer, ctx_docs, mem_docs = adaptive_answer(text, history=history, verbose=True)

    # Answer
    print(f"\n  A: {answer}")

    # Add to history
    history.append(AIMessage(content=answer))

    # History snapshot
    print(f"\n  📝 HISTORY SNAPSHOT ({len(history)} messages):")
    for msg in history:
        role    = "You  " if isinstance(msg, HumanMessage) else "Askly"
        snippet = msg.content[:70].replace("\n", " ")
        ellipsis = "..." if len(msg.content) > 70 else ""
        print(f"      {role}: {snippet}{ellipsis}")

print(f"\n{'=' * 70}")
print("✅ Adaptive RAG demo complete.")


🤖 ADAPTIVE RAG — Scripted Demo

──────────────────────────────────────────────────────────────────────
  Turn 1  |  History: 0 messages
  Q: What is ashwagandha?
──────────────────────────────────────────────────────────────────────
  🔀 ROUTER: strategy=dense, k=12, alpha=N/A
     Notes: User is asking for a definition of a specific term. Dense retrieval is suitable for direct factual recall.
  📊 RETRIEVAL: 20 corpus docs, 0 memory docs
  📈 TOKEN BUDGET: 13 corpus docs kept (1159/1841 tokens, 62%)

  A: Ashwagandha is an herb widely used in Indian traditions [S1], [S4]. It is often combined with roasted coffee in a modern fusion trend [S1], [S4]. It is considered an adaptogen [S12] and has an earthy, slightly bitter taste [S3].

  📝 HISTORY SNAPSHOT (2 messages):
      You  : What is ashwagandha?
      Askly: Ashwagandha is an herb widely used in Indian traditions [S1], [S4]. It...

──────────────────────────────────────────────────────────────────────
  Turn 2  |  History: 2 messages